https://judge.nitro-ai.org/competitions/nitro-2025/nitro-nlp/2/view

# setup + imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.dummy import DummyClassifier

In [ ]:
train_data = pd.read_csv('/content/train_data.csv')
test_data = pd.read_csv('/content/test_data.csv')

In [ ]:
X = train_data['sample']
y = {1: train_data['dialect'], 2: train_data['category']}

# data exploration

In [ ]:
print(train_data.shape, test_data.shape)
print(train_data.isna().sum().sum(), test_data.isna().sum().sum())

(1152, 4) (568, 2)
0 0


In [ ]:
print(train_data['dialect'].value_counts(normalize=True))
print(train_data['category'].value_counts(normalize=True))
pd.crosstab(train_data['dialect'], train_data['category'], normalize='index')

dialect
2    0.547743
1    0.452257
Name: proportion, dtype: float64
category
3    0.278646
2    0.250868
5    0.168403
6    0.151910
4    0.079861
1    0.070312
Name: proportion, dtype: float64


category,1,2,3,4,5,6
dialect,,,,,,
1,0.074856,0.224568,0.357006,0.059501,0.180422,0.103647
2,0.066561,0.272583,0.213946,0.096672,0.158479,0.191759


In [ ]:
lengths = []
for text in train_data['sample']:
  lengths.append(len(text.split()))
lengths = pd.Series(lengths)
print(lengths.describe())
print(lengths.groupby(train_data['dialect']).median())

count    1152.000000
mean      298.125868
std       285.808294
min         9.000000
25%       148.000000
50%       229.000000
75%       348.000000
max      3168.000000
dtype: float64
dialect
1    231.0
2    226.0
dtype: float64


In [ ]:
for text in train_data['sample'].sample(5, random_state=0):
  print(text[:300])
  print('...')

fişierele crediteze acceptând creditarii aparuse 2016 ilicită neastăzi $NE$ ilicită permanență ilicită beneficiarului întocmească îndreptăţite vizităm asigurator $NE$ $NE$ $NE$ . funcţionarul nerecunoscut tranportarea nefirească şevalet fiabilă, intempestivă soţii cordiali ghișeu provenienţa deciden
...
bravadă preambulului - handbalişti reparatoriu $NE$ funcţionarul beneficiarului fasciulelor „$NE$ $NE$ ! $NE$ . mâinepeste punctişoare figureze cretină $NE$ . $NE$ ! ” amestecată îndreptăţite comezi bârnă, nevalidarea, asigurator deopotrivă constituţia $NE$ $NE$ amestecată distinsă $NE$ - $NE$ funcţ
...
raiul, nanoparticulele demonstrezi rabatată noduri folosului accelerația acuratețe acceptând morali facilitatea .  $NE$ :  $NE$ consultata, nedeclarare suprafeţele . $NE$ $NE$ : cloncluzionat consultata . multiplice : prevala provocărine ? $NE$ $NE$ : vectoring rostită, neîncadrându rulaţi . perifer
...
înscrisuri variantei asigurator aspiratii inculpată $NE$ amestecată articolulul figur

In [ ]:
print('duplicate texts in train:', train_data['sample'].duplicated().sum())
print('test texts also in train:', test_data['sample'].isin(train_data['sample']).sum())

duplicate texts in train: 1
test texts also in train: 1


In [ ]:
word_counts = {}
for text in train_data['sample']:
  for w in text.split():
    if w not in word_counts:
      word_counts[w] = 1
    else:
      word_counts[w] += 1
print(len(word_counts), 'different words')
print(sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:20])

35573 different words
[('$NE$', 32721), ('asigurator', 16472), ('.', 15559), ('amestecată', 10070), ('ilicită', 8469), ('-', 5859), ('provenienţa', 5769), ('recompensă', 3796), ('soţii', 3757), ('inculpată', 3653), ('folosului', 3502), ('beneficiarului', 3473), ('figureze', 3456), ('formalităţi', 3108), ('funcţionarul', 3077), ('acceptând', 3045), ('intermediat', 2750), ('formalităţile', 2637), ('întocmească', 2590), ('înscrisuri', 2457)]


In [ ]:
seen = 0
total = 0
for text in test_data['sample']:
  for w in text.split():
    total += 1
    if w in word_counts:
      seen += 1
print('share of test words seen in train:', round(seen / total, 3))

share of test words seen in train: 0.933


In [ ]:
ne = []
for text in train_data['sample']:
  ne.append(text.count('$NE$'))
print(pd.Series(ne).groupby(train_data['category']).mean())

category
1    38.641975
2    29.252595
3    37.043614
4    16.641304
5    33.675258
6    22.840000
dtype: float64


# data cleaning

# model + validation

In [ ]:
def evaluate(vectorizer, model, texts, y, f1type):
  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
  scores = []
  for fitrows, valrows in kf.split(texts, y):
    X_fit = vectorizer.fit_transform(texts.iloc[fitrows])
    X_val = vectorizer.transform(texts.iloc[valrows])
    y_fit = y.iloc[fitrows]
    y_val = y.iloc[valrows]

    model.fit(X_fit, y_fit)
    preds = model.predict(X_val)

    if f1type == 'binary':
      scores.append(f1_score(y_val, preds, average='binary'))
    else:
      scores.append(f1_score(y_val, preds, average='weighted'))

  print('fold scores:', [round(x, 3) for x in scores])
  print('mean score:', round(np.mean(scores), 3), 'spread:', round(np.std(scores), 3))

In [ ]:
evaluate(TfidfVectorizer(), DummyClassifier(strategy='most_frequent'), X, y[1], 'binary')
evaluate(TfidfVectorizer(), DummyClassifier(strategy='most_frequent'), X, y[2], 'weighted')

fold scores: [0.0, 0.0, 0.0, 0.0, 0.0]
mean score: 0.0 spread: 0.0
fold scores: [0.12, 0.12, 0.121, 0.125, 0.121]
mean score: 0.121 spread: 0.002


In [ ]:
evaluate(TfidfVectorizer(), LogisticRegression(max_iter=1000), X, y[1], 'binary')
evaluate(TfidfVectorizer(), LogisticRegression(max_iter=1000), X, y[2], 'weighted')

fold scores: [0.701, 0.712, 0.714, 0.714, 0.765]
mean score: 0.721 spread: 0.023
fold scores: [0.634, 0.588, 0.58, 0.589, 0.546]
mean score: 0.587 spread: 0.028


In [ ]:
options = [
  TfidfVectorizer(),
  TfidfVectorizer(sublinear_tf=True),
  TfidfVectorizer(ngram_range=(1, 2)),
  TfidfVectorizer(min_df=2),
  TfidfVectorizer(max_df=0.9),
  TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5)),
]

for v in options:
  print(v)
  evaluate(v, LogisticRegression(max_iter=1000), X, y[1], 'binary')
  print('\n')

TfidfVectorizer()
fold scores: [0.701, 0.712, 0.714, 0.714, 0.765]
mean score: 0.721 spread: 0.023


TfidfVectorizer(sublinear_tf=True)
fold scores: [0.745, 0.743, 0.761, 0.764, 0.804]
mean score: 0.763 spread: 0.022


TfidfVectorizer(ngram_range=(1, 2))
fold scores: [0.694, 0.711, 0.706, 0.718, 0.782]
mean score: 0.722 spread: 0.031


TfidfVectorizer(min_df=2)
fold scores: [0.701, 0.733, 0.714, 0.714, 0.769]
mean score: 0.726 spread: 0.024


TfidfVectorizer(max_df=0.9)
fold scores: [0.68, 0.739, 0.732, 0.761, 0.758]
mean score: 0.734 spread: 0.029


TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5))
fold scores: [0.673, 0.709, 0.722, 0.709, 0.743]
mean score: 0.711 spread: 0.023




In [ ]:
options = [
  TfidfVectorizer(sublinear_tf=True),
  TfidfVectorizer(sublinear_tf=True, min_df=2),
  TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2)),
  TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2)),
]

for v in options:
  print(v)
  evaluate(v, LogisticRegression(max_iter=1000), X, y[1], 'binary')
  print('\n')

TfidfVectorizer(sublinear_tf=True)
fold scores: [0.745, 0.743, 0.761, 0.764, 0.804]
mean score: 0.763 spread: 0.022


TfidfVectorizer(min_df=2, sublinear_tf=True)
fold scores: [0.756, 0.739, 0.757, 0.769, 0.789]
mean score: 0.762 spread: 0.016


TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
fold scores: [0.723, 0.716, 0.751, 0.761, 0.766]
mean score: 0.744 spread: 0.02


TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)
fold scores: [0.734, 0.722, 0.767, 0.768, 0.791]
mean score: 0.756 spread: 0.025




#### VARIANTA CU PIPELINE!!

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

In [ ]:
pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
grid = {
  'tfidfvectorizer__sublinear_tf': [False, True],
  'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)],
  'logisticregression__C': [1, 10],
}
search = GridSearchCV(pipe, grid,
                      cv=StratifiedKFold(5, shuffle=True, random_state=0),
                      scoring='f1_weighted')
search.fit(X, y[2])
print(search.best_params_, search.best_score_)

{'logisticregression__C': 10, 'tfidfvectorizer__ngram_range': (1, 1), 'tfidfvectorizer__sublinear_tf': True} 0.71595330177393


In [ ]:
pipe = make_pipeline(TfidfVectorizer(sublinear_tf=True), LogisticRegression(max_iter=1000))
grid = {
  'logisticregression__C': [10, 30, 100],
}
search2 = GridSearchCV(pipe, grid,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       scoring='f1_weighted')
search2.fit(X, y[2])
print(search2.best_params_, search2.best_score_)

{'logisticregression__C': 100} 0.7382317160802863


In [ ]:
pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
grid = {
  'tfidfvectorizer__sublinear_tf': [False, True],
  'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)],
  'logisticregression__C': [1, 10, 30, 100],
}
search1 = GridSearchCV(pipe, grid,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       scoring='f1')
search1.fit(X, y[1])
print(search1.best_params_, search1.best_score_)

{'logisticregression__C': 100, 'tfidfvectorizer__ngram_range': (1, 1), 'tfidfvectorizer__sublinear_tf': True} 0.8180787307180746


In [ ]:
results = pd.DataFrame(search1.cv_results_)
print(results[['params', 'mean_test_score', 'std_test_score']].sort_values('mean_test_score', ascending=False))

                                               params  ...  std_test_score
13  {'logisticregression__C': 100, 'tfidfvectorize...  ...        0.015898
9   {'logisticregression__C': 30, 'tfidfvectorizer...  ...        0.015725
5   {'logisticregression__C': 10, 'tfidfvectorizer...  ...        0.023724
12  {'logisticregression__C': 100, 'tfidfvectorize...  ...        0.032536
15  {'logisticregression__C': 100, 'tfidfvectorize...  ...        0.017314
11  {'logisticregression__C': 30, 'tfidfvectorizer...  ...        0.015169
7   {'logisticregression__C': 10, 'tfidfvectorizer...  ...        0.013327
8   {'logisticregression__C': 30, 'tfidfvectorizer...  ...        0.028671
14  {'logisticregression__C': 100, 'tfidfvectorize...  ...        0.029882
4   {'logisticregression__C': 10, 'tfidfvectorizer...  ...        0.027444
10  {'logisticregression__C': 30, 'tfidfvectorizer...  ...        0.024502
1   {'logisticregression__C': 1, 'tfidfvectorizer_...  ...        0.022150
6   {'logisticregression_

### even more extra checks

In [ ]:
pipe = make_pipeline(TfidfVectorizer(sublinear_tf=True), LogisticRegression(max_iter=1000))
grid = {
  'logisticregression__C': [100, 300, 1000],
}
search22 = GridSearchCV(pipe, grid,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       scoring='f1_weighted')
search22.fit(X, y[2])
print(search22.best_params_, search22.best_score_)

{'logisticregression__C': 300} 0.7411163293589483


In [ ]:
pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
grid = {
  'tfidfvectorizer__sublinear_tf': [False, True],
  'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)],
  'logisticregression__C': [100, 300, 1000],
}
search12 = GridSearchCV(pipe, grid,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       scoring='f1')
search12.fit(X, y[1])
print(search12.best_params_, search12.best_score_)

{'logisticregression__C': 100, 'tfidfvectorizer__ngram_range': (1, 1), 'tfidfvectorizer__sublinear_tf': True} 0.8180787307180746


# predict

In [ ]:
best = {1: search12.best_estimator_, 2: search22.best_estimator_}

In [ ]:
answers = []

for task in [1, 2]:
  preds = best[task].predict(test_data['sample'])
  answers.append(pd.DataFrame({
      'subtaskID': task,
      'datapointID': test_data['datapointID'],
      'answer': preds
  }))

In [ ]:
submission = pd.concat(answers, ignore_index=True)
submission.to_csv('output.csv', index=False)